# 허깅페이스 다국어 번역 모델 받아 번역기 만드르기
[https://huggingface.co/facebook/m2m100_418M](https://huggingface.co/facebook/m2m100_418M)

In [4]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

text = "जीवन एक चॉकलेट बॉक्स की तरह है।"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate Hindi to French
tokenizer.src_lang = "hi"  
encoded_hi = tokenizer(hi_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("ko"))
result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
# => "La vie est comme une boîte de chocolat."

print(result)

Loading weights: 100%|█████████████████████████████████████████| 512/512 [00:00<00:00, 15620.90it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['인생은 초콜릿 상자와 같다.']


In [5]:
import gradio as gr
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 로드
MODEL_NAME = "facebook/m2m100_418M"
model = M2M100ForConditionalGeneration.from_pretrained(MODEL_NAME)
tokenizer = M2M100Tokenizer.from_pretrained(MODEL_NAME)

# 사용 가능한 언어 매핑
LANGUAGE_MAP = {
    "언어감지": "auto",
    "한국어": "ko",
    "영어": "en",
    "일본어": "ja",
    "중국어(간체)": "zh",
    "중국어(번체)": "zh_tw",
    "힌디어": "hi",
    "프랑스어": "fr",
    "독일어": "de",
    "스페인어": "es",
    "러시아어": "ru",
    "베트남어": "vi",
    "태국어": "th",
    "인도네시아어": "id",
    "아랍어": "ar",
}

# 간단한 언어 감지 함수
def detect_language(text: str):
    if not text or not text.strip():
        return "en"

    # 매우 단순한 문자 기반 감지
    for ch in text:
        code = ord(ch)
        if 0xAC00 <= code <= 0xD7A3:
            return "ko"   # 한글
        elif 0x0900 <= code <= 0x097F:
            return "hi"   # 데바나가리(힌디어)
        elif 0x3040 <= code <= 0x30FF:
            return "ja"   # 일본어
        elif 0x4E00 <= code <= 0x9FFF:
            return "zh"   # 한자권 간단 처리
        elif 0x0600 <= code <= 0x06FF:
            return "ar"   # 아랍어
        elif 0x0400 <= code <= 0x04FF:
            return "ru"   # 키릴 문자

    return "en"

def translate_text(text, src_label, tgt_label):
    if not text or not text.strip():
        return ""

    src_lang = LANGUAGE_MAP.get(src_label, "auto")
    tgt_lang = LANGUAGE_MAP.get(tgt_label, "ko")

    if src_lang == "auto":
        src_lang = detect_language(text)

    try:
        tokenizer.src_lang = src_lang
        encoded = tokenizer(text, return_tensors="pt", truncation=True)

        with torch.no_grad():
            generated_tokens = model.generate(
                **encoded,
                forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
                max_length=256
            )

        result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        return result[0] if result else ""
    except Exception as e:
        return f"번역 중 오류가 발생했습니다: {e}"

def swap_languages(src_label, tgt_label, input_text, output_text):
    return tgt_label, src_label, output_text, input_text

custom_css = """
body {
    background: #f5f5f5;
}
.gradio-container {
    max-width: 1280px !important;
    margin: 30px auto !important;
    font-family: 'Noto Sans KR', sans-serif;
}
.top-tabs {
    display: flex;
    justify-content: center;
    margin-bottom: 18px;
}
.tab-pill {
    background: white;
    border: 1px solid #d9d9d9;
    border-radius: 999px;
    padding: 8px 18px;
    font-size: 15px;
    color: #444;
}
.panel-wrap {
    gap: 12px !important;
}
.box-panel {
    border: 1px solid #dcdcdc !important;
    border-radius: 18px !important;
    background: white !important;
    box-shadow: none !important;
}
textarea {
    font-size: 18px !important;
}
.primary-btn {
    background: #21c55d !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    height: 56px !important;
}
.swap-btn {
    min-width: 52px !important;
    height: 52px !important;
    border-radius: 999px !important;
    font-size: 20px !important;
}
"""

with gr.Blocks(css=custom_css, title="번역기") as demo:
    gr.HTML("""
    <div class="top-tabs">
        <div class="tab-pill">T 텍스트 &nbsp;&nbsp; 🖼 이미지 &nbsp;&nbsp; 📄 문서 &nbsp;&nbsp; 🌐 웹사이트</div>
    </div>
    """)

    with gr.Row(elem_classes="panel-wrap"):
        with gr.Column(scale=1, elem_classes="box-panel"):
            src_lang = gr.Dropdown(
                choices=list(LANGUAGE_MAP.keys()),
                value="언어감지",
                label=None,
                interactive=True
            )
            input_text = gr.Textbox(
                placeholder="번역할 내용을 입력하세요",
                lines=12,
                show_label=False
            )

        with gr.Column(scale=0, min_width=70):
            gr.Markdown("")
            swap_btn = gr.Button("⇄", elem_classes="swap-btn")

        with gr.Column(scale=1, elem_classes="box-panel"):
            tgt_lang = gr.Dropdown(
                choices=list(LANGUAGE_MAP.keys())[1:],
                value="한국어",
                label=None,
                interactive=True
            )
            output_text = gr.Textbox(
                placeholder="번역 결과",
                lines=12,
                show_label=False,
                interactive=False
            )

    with gr.Row():
        translate_btn = gr.Button("번역하기", elem_classes="primary-btn")

    translate_btn.click(
        fn=translate_text,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=output_text
    )

    input_text.submit(
        fn=translate_text,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=output_text
    )

    swap_btn.click(
        fn=swap_languages,
        inputs=[src_lang, tgt_lang, input_text, output_text],
        outputs=[src_lang, tgt_lang, input_text, output_text]
    )

demo.launch()

Loading weights: 100%|██████████████████████████████████████████| 512/512 [00:00<00:00, 8900.71it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/tmp/ipykernel_2408/1514880492.py:131: UserWarning: The parameters have been moved f

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
demo.close()

# 한영-영한 번역 결과를 음성으로 읽어주는 서비스

In [ ]:
# !pip install "transformers>=4.33" accelerate soundfile
# !pip install "kokoro>=0.9.2"    # 한국어 로마자 변환기
# !pip install uroman  # 전 세계 언어 로마자 변환기
# !sudo apt-get -y install espeak-ng

In [1]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    VitsModel,
    AutoTokenizer,
)
from kokoro import KPipeline   # Kokoro-82M 파이썬 래퍼
from uroman import Uroman      # 🔥 pip install uroman

# -----------------------------
# 0. 디바이스 (GPU 우선)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. 번역 모델 (M2M100)
# -----------------------------
translation_model_name = "facebook/m2m100_418M"
translation_model = M2M100ForConditionalGeneration.from_pretrained(
    translation_model_name
).to(device)
translation_tokenizer = M2M100Tokenizer.from_pretrained(translation_model_name)

# -----------------------------
# 2. 영어 TTS: Kokoro-82M
# -----------------------------
kokoro_pipeline = KPipeline(lang_code="a")

def tts_english_kokoro(text: str, path: str = "tts_en.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    generator = kokoro_pipeline(text, voice="af_heart")

    chunks = []
    for _, _, audio in generator:
        chunks.append(audio)

    if not chunks:
        return None

    audio_cat = np.concatenate(chunks)
    sf.write(path, audio_cat, 24000)
    return path

# -----------------------------
# 3. 한국어 TTS: MMS-TTS-KOR (VITS)
#    🔥 pip uroman 사용
# -----------------------------
kor_tts_name = "facebook/mms-tts-kor"
kor_tts_model = VitsModel.from_pretrained(kor_tts_name).to(device)
kor_tts_tokenizer = AutoTokenizer.from_pretrained(kor_tts_name)

uroman = Uroman()   # 🔥 한 번만 초기화하면 됨

def tts_korean_mms(text: str, path: str = "tts_ko.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    # 1) 🔥 uroman 로마자 변환 (찾는 발음형)
    roman_text = uroman.romanize_string(text)

    # 2) 토크나이저 입력
    inputs = kor_tts_tokenizer(roman_text, return_tensors="pt").to(device)

    # 3) 모델 인퍼런스
    with torch.no_grad():
        outputs = kor_tts_model(**inputs).waveform

    wav = outputs.squeeze().cpu().numpy()
    if wav.size == 0:
        return None

    sf.write(path, wav, samplerate=kor_tts_model.config.sampling_rate)
    return path

# -----------------------------
# 4. 언어 매핑
# -----------------------------
LANGUAGES = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Hindi": "hi",
}

# -----------------------------
# 5. 번역 + TTS 함수
# -----------------------------
def translate_and_speak(text, src_lang_name, tgt_lang_name):
    text = text or ""
    if not text.strip():
        return "", None

    src_lang = LANGUAGES[src_lang_name]
    tgt_lang = LANGUAGES[tgt_lang_name]

    # ----- 번역 -----
    if src_lang == tgt_lang:
        translated_text = text.strip()
    else:
        translation_tokenizer.src_lang = src_lang
        encoded = translation_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            generated_tokens = translation_model.generate(
                **encoded,
                forced_bos_token_id=translation_tokenizer.get_lang_id(tgt_lang),
                max_length=512,
            )

        translated_text = translation_tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True
        )[0]

    # ----- TTS -----
    audio_path = None
    if tgt_lang == "en":
        audio_path = tts_english_kokoro(translated_text)
    elif tgt_lang == "ko":
        audio_path = tts_korean_mms(translated_text)

    return translated_text, audio_path

# -----------------------------
# 6. Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역 + TTS (영어 Kokoro + 한국어 MMS)")

    with gr.Row():
        src_lang = gr.Dropdown(list(LANGUAGES.keys()), label="원본 언어", value="Korean")
        tgt_lang = gr.Dropdown(list(LANGUAGES.keys()), label="번역 언어", value="English")

    with gr.Row():
        input_text = gr.Textbox(lines=8, label="입력 텍스트")
        output_text = gr.Textbox(lines=8, label="번역 결과", interactive=False)

    audio_output = gr.Audio(label="음성 출력", type="filepath")

    translate_button = gr.Button("번역 + 음성 생성")
    translate_button.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, audio_output],
    )

    demo.queue()

demo.launch(inline=False, share=False)


/home/ksjin/miniforge3/envs/hug/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ksjin/miniforge3/envs/hug/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|█████████████████████████████████████████| 512/512 [00:00<00:00, 15255.59it/s]
The tied weights mapping and config for this model specifies to tie m

/home/ksjin/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/ksjin/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Loading weights: 100%|██████████████████████████████████████████| 762/762 [00:00<00:00, 1722.94it/s]


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
